In [ ]:
pip install rapidfuzz pandas

In [ ]:
import re
import unicodedata
import pandas as pd
from pathlib import Path
from rapidfuzz import fuzz

In [ ]:
mis_form_path = "../../backend/source/forms"
flow_form_path = "./output/flow_forms"

flow_ids = {
    "8520967": 1749634736797, # WTP
    "17260923": 1748903240763, # WWTP
    "27040920": 1749611049520, # SPS (Pump Station)
    "1520924": 1749623934933, # EPS Water quality
    "5530933": 1749623934933, # EPS Project Construction
    "2490944": 1749621221728, # RWS
}

flow_mis_child_forms = {
    "8520967": [
        1749640508297,
        1749652214711,
    ],
    "17260923": [
        1748905550055,
        1748918946591,
    ],
    "27040920": [
        1749611905372,
        1749627302948,
    ],
    "1520924": [
        1749632545233,
    ],
    "5530933": [
        1749624452908,
    ],
    "2490944": [
        1749621962296,
        1749631041125,
    ]
}

meta_config = {
    "flow": [
        "name",
        "surveyId",
        "surveyGroupId",
        "version",
        "app",
        ["questionGroup", "heading"],
        ["questionGroup", "repeatable"],
    ],
    "mis": [
        "id",
        "form",
        "type",
        "version",
        ["question_groups", "label"],
        ["question_groups", "repeatable"],
        ["question_groups", "order"],
    ],
}

# Configuration - Text-only matching (ignoring question groups completely)
SIMILARITY_CONFIG = {
    "text_similarity_threshold": 80,  # Minimum score for text match (0-100)
    "use_group_weighting": False,     # Completely ignore question groups
    "group_weight": 0.0,              # No group weighting
    "text_weight": 1.0,               # Use only text similarity
}

In [ ]:
def flatten_flow_form_pd(
    json_path: Path,
    record_path: str = "questionGroup.question",
    meta_type: str = "flow",
) -> pd.DataFrame:
    if meta_type not in meta_config:
        raise ValueError(f"meta_type must be one of {list(meta_config.keys())}")
    # read top-level JSON as a pandas Series (one object)
    j = pd.read_json(json_path, typ="series").to_dict()

    # normalize nested questions: record_path points to question lists inside questionGroup
    df = pd.json_normalize(
        j,
        record_path=record_path.split("."),
        meta=meta_config[meta_type],
        meta_prefix=f"{meta_type}_",
        errors="ignore"
    )
    # rename question id column for clarity
    if "id" in df.columns:
        df = df.rename(
            columns={
                "id": f"{meta_type}_question_id"
            }
        )

    # rename group meta columns for clarity
    if "flow_questionGroup.heading" in df.columns:
        df = df.rename(
            columns={
                "flow_questionGroup.heading": "flow_question_group",
                "flow_questionGroup.repeatable": "flow_question_group_repeatable"
            }
        )

    if "mis_question_groups.label" in df.columns:
        df = df.rename(
            columns={
                "mis_question_groups.label": "mis_question_group",
                "mis_question_groups.repeatable": "mis_question_group_repeatable",
                "mis_question_groups.order": "mis_question_group_order"
            }
        )

    # Rename question order column for MIS forms
    if meta_type == "mis" and "order" in df.columns:
        df = df.rename(columns={"order": "mis_question_order_raw"})

    return df

In [ ]:
def _normalize_alpha_only(val):
    if pd.isna(val):
        return ""
    s = str(val)
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii")
    s = s.lower()
    # remove anything that's not a-z (keeps only letters)
    s = re.sub(r"[^a-z]+", "", s)
    return s

In [ ]:
def _normalize_for_similarity(val):
    """
    Normalize text for similarity matching.
    Keeps more structure than _normalize_alpha_only for better fuzzy matching.
    """
    if pd.isna(val):
        return ""
    s = str(val)
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii")
    s = s.lower()
    # Remove extra whitespace and normalize punctuation
    s = re.sub(r'\s+', ' ', s)
    s = s.strip()
    return s

In [ ]:
def calculate_text_similarity(text1: str, text2: str) -> float:
    """
    Calculate text similarity score using rapidfuzz.
    Uses partial_ratio which is good for matching substrings.
    
    This function completely ignores question groups.
    
    Args:
        text1: First text string (Flow question label)
        text2: Second text string (MIS question label)
    
    Returns:
        Similarity score between 0 and 100
    """
    norm_text1 = _normalize_for_similarity(text1)
    norm_text2 = _normalize_for_similarity(text2)
    
    # Use partial_ratio for better matching of questions with slight variations
    # Example: "Which Division-Province-Tikina are you in?" vs "Which Division-Province-Tikina?"
    text_score = fuzz.partial_ratio(norm_text1, norm_text2)
    
    return text_score

In [ ]:
def find_best_match_by_text(
    flow_text: str,
    mis_questions_df: pd.DataFrame,
    config: dict = SIMILARITY_CONFIG
) -> tuple[pd.DataFrame, dict]:
    """
    Find the best matching MIS question for a Flow question based ONLY on text similarity.
    Question groups are completely ignored.
    
    Args:
        flow_text: Flow question label text
        mis_questions_df: DataFrame containing MIS questions
        config: Configuration dictionary
    
    Returns:
        tuple: (matched_row, match_info)
    """
    if mis_questions_df.empty:
        return pd.DataFrame(), {"matched": False, "reason": "No MIS questions available"}
    
    # Get all MIS question labels
    mis_labels = mis_questions_df["label"].tolist()
    
    best_score = 0
    best_idx = None
    
    # Iterate through all MIS questions to find best text match
    for idx, mis_label in enumerate(mis_labels):
        score = calculate_text_similarity(flow_text, mis_label)
        
        if score > best_score:
            best_score = score
            best_idx = idx
    
    # Check if best score meets threshold
    if best_score >= config["text_similarity_threshold"]:
        match_info = {
            "matched": True,
            "score": best_score,
            "method": "text_similarity"
        }
        return mis_questions_df.iloc[[best_idx]], match_info
    else:
        match_info = {
            "matched": False,
            "score": best_score,
            "reason": f"Score {best_score:.1f} below threshold {config['text_similarity_threshold']}"
        }
        return pd.DataFrame(), match_info

In [ ]:
def match_questions_by_text(
    flow_questions_df: pd.DataFrame,
    mis_questions_df: pd.DataFrame,
    config: dict = SIMILARITY_CONFIG
) -> pd.DataFrame:
    """
    Match Flow questions to MIS questions using text-only similarity matching.
    Question groups are completely ignored.
    
    This function replaces the exact matching logic with a fuzzy matching approach
    that works on question labels only.
    """
    rows = []
    unmatched_count = 0
    low_confidence_count = 0
    
    # Iterate every unique flow question row grouped by surveyId, group, question id
    for (survey_id, group, qid), grp in flow_questions_df.groupby(
        ["flow_surveyId", "flow_question_group", "flow_question_id"], dropna=False
    ):
        rep = grp.iloc[0]
        f_text = rep.get("text", "")
        f_group = rep.get("flow_question_group", "")
        
        # Find best match using text similarity only (ignoring groups)
        matched_df, match_info = find_best_match_by_text(f_text, mis_questions_df, config)
        
        if match_info["matched"]:
            # Collect aggregated values from matched rows
            def agg_col(df, col):
                if col not in df.columns or df.empty:
                    return ""
                vals = df[col].dropna().astype(str).unique()
                return ";".join(vals)
            
            mis_id_val = agg_col(matched_df, "mis_id")
            mis_qgroup_val = agg_col(matched_df, "mis_question_group")
            mis_qid_val = agg_col(matched_df, "mis_question_id")
            label_val = agg_col(matched_df, "label")
            
            # Calculate mis_question_order from group order and question order
            # Formula: group_order * 1000 + question_order
            mis_question_order_val = ""
            if "mis_question_group_order" in matched_df.columns and "mis_question_order_raw" in matched_df.columns:
                matched_row = matched_df.iloc[0]
                group_order = matched_row.get("mis_question_group_order")
                question_order = matched_row.get("mis_question_order_raw")
                if pd.notna(group_order) and pd.notna(question_order):
                    mis_question_order_val = int(group_order) * 1000 + int(question_order)
            
            # Determine confidence level based on score
            score = match_info["score"]
            if score >= 95:
                confidence = "high"
            elif score >= 85:
                confidence = "medium"
            else:
                confidence = "low"
                low_confidence_count += 1
            
            rows.append({
                "flow_form_id": survey_id,
                "flow_question_group": group,
                "flow_question_label": f_text,
                "flow_question_id": qid,
                "mis_form_id": mis_id_val,
                "mis_question_group": mis_qgroup_val,
                "mis_question_label": label_val,
                "mis_question_id": mis_qid_val,
                "match_score": round(score, 2),
                "match_confidence": confidence,
                "match_method": match_info["method"],
                "mis_question_order": mis_question_order_val
            })
        else:
            # No match found - still create a row with empty MIS values
            rows.append({
                "flow_form_id": survey_id,
                "flow_question_group": group,
                "flow_question_label": f_text,
                "flow_question_id": qid,
                "mis_form_id": "",
                "mis_question_group": "",
                "mis_question_label": "",
                "mis_question_id": "",
                "match_score": round(match_info.get("score", 0), 2),
                "match_confidence": "none",
                "match_method": "none",
                "mis_question_order": ""
            })
            unmatched_count += 1
    
    mapping_df = pd.DataFrame(rows)
    
    # Print summary statistics
    total_questions = len(mapping_df)
    matched_questions = total_questions - unmatched_count
    print(f"\n=== Matching Summary ===")
    print(f"Total Flow questions: {total_questions}")
    print(f"Successfully matched: {matched_questions} ({matched_questions/total_questions*100:.1f}%)")
    print(f"Unmatched: {unmatched_count} ({unmatched_count/total_questions*100:.1f}%)")
    print(f"Low confidence matches: {low_confidence_count}")
    print(f"Threshold: {config['text_similarity_threshold']}%")
    
    return mapping_df

In [ ]:
def find_mis_form_by_id(form_id: int) -> dict | None:
    """
    Find a MIS form JSON file by its form ID.
    
    Args:
        form_id: The MIS form ID to find
    
    Returns:
        Dictionary with form info or None if not found:
        {'id': 1749623934933, 'name': 'EPS Registration', 'file': Path(...)}
    """
    for json_file in Path(mis_form_path).iterdir():
        if json_file.suffix != ".json":
            continue
        
        form_data = pd.read_json(json_file, typ="series").to_dict()
        if form_data.get("id") == form_id:
            return {
                'id': form_id,
                'name': form_data.get('form', ''),
                'file': json_file
            }
    
    return None


def convert_float_ids_to_int(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """
    Convert float ID columns to integer strings, preserving empty values.
    This fixes the issue where pandas reads numeric columns with NaN as float.
    
    Args:
        df: DataFrame to process
        columns: List of column names to convert
    
    Returns:
        DataFrame with converted columns
    """
    df = df.copy()
    for col in columns:
        if col in df.columns:
            # Convert float to int string, preserving empty/NaN values
            df[col] = df[col].apply(
                lambda x: str(int(x)) if pd.notna(x) and x != '' else ''
            )
    return df


def load_existing_manual_matches(output_csv: Path) -> pd.DataFrame:
    """
    Load existing manual matches from a CSV file.
    
    Args:
        output_csv: Path to the existing CSV file
    
    Returns:
        DataFrame containing only rows with match_method = 'manual',
        or empty DataFrame if file doesn't exist or has no manual matches
    """
    if not output_csv.exists():
        return pd.DataFrame()
    
    try:
        # Read CSV with ID columns as strings to avoid float conversion
        existing_df = pd.read_csv(
            output_csv,
            dtype={
                'mis_question_id': str,
                'mis_form_id': str,
                'flow_form_id': str,
                'flow_question_id': str,
                'mis_question_order': str,
            }
        )
        # Filter only manual matches
        manual_df = existing_df[existing_df.get('match_method', '') == 'manual'].copy()
        # Convert flow_form_id to string to match new_mapping_df dtype
        if not manual_df.empty and 'flow_form_id' in manual_df.columns:
            manual_df['flow_form_id'] = manual_df['flow_form_id'].astype(str)
        # Clean up any remaining float-like strings (e.g., "1749693489743.0" -> "1749693489743")
        id_columns = ['mis_question_id', 'mis_form_id', 'mis_question_order']
        for col in id_columns:
            if col in manual_df.columns:
                manual_df[col] = manual_df[col].apply(
                    lambda x: str(int(float(x))) if pd.notna(x) and x != '' and x != 'nan' else ''
                )
        return manual_df
    except Exception as e:
        print(f"Warning: Could not load existing CSV {output_csv}: {e}")
        return pd.DataFrame()


def merge_manual_matches(
    new_mapping_df: pd.DataFrame,
    manual_matches_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Merge new auto-generated mappings with existing manual matches.
    Manual matches take precedence over auto-generated matches.
    
    Args:
        new_mapping_df: Newly generated mapping DataFrame
        manual_matches_df: Existing manual matches DataFrame
    
    Returns:
        Merged DataFrame with manual matches preserved
    """
    if manual_matches_df.empty:
        return new_mapping_df
    
    # Ensure both DataFrames have the same dtype for merge keys
    manual_matches_df = manual_matches_df.copy()
    manual_matches_df['flow_form_id'] = manual_matches_df['flow_form_id'].astype(str)
    manual_matches_df['flow_question_id'] = manual_matches_df['flow_question_id'].astype(str)
    
    new_mapping_df = new_mapping_df.copy()
    new_mapping_df['flow_form_id'] = new_mapping_df['flow_form_id'].astype(str)
    new_mapping_df['flow_question_id'] = new_mapping_df['flow_question_id'].astype(str)
    
    # Remove rows from new_mapping_df that have manual matches in manual_matches_df
    # Match on flow_form_id and flow_question_id
    manual_keys = manual_matches_df[['flow_form_id', 'flow_question_id']].drop_duplicates()
    
    # Filter out rows that have manual matches
    merged_df = new_mapping_df.merge(
        manual_keys,
        on=['flow_form_id', 'flow_question_id'],
        how='left',
        indicator=True
    )
    
    # Keep rows that don't have manual matches (left_only) + add all manual matches
    auto_only = merged_df[merged_df['_merge'] == 'left_only'].drop(columns=['_merge'])
    
    # Combine auto-generated matches with preserved manual matches
    result_df = pd.concat([auto_only, manual_matches_df], ignore_index=True)
    
    return result_df

In [ ]:
# Main processing loop
output_dir = Path("./output/forms")

# Columns that should be saved as integers (not floats)
ID_COLUMNS = ['mis_question_id', 'mis_form_id', 'mis_question_order']

for json_form in filter(
    lambda f: f.suffix == ".json",
    Path(flow_form_path).iterdir(),
):
    print(f"\n{'='*60}")
    print(f"Loading form from {json_form}")
    
    # Get filename and extract form id
    flow_form_id = json_form.name.split("_")[0]
    print(f"Form ID: {flow_form_id}")
    
    # Load Flow questions
    flow_questions_df = flatten_flow_form_pd(json_form)
    
    # Get MIS parent form id from mapping
    parent_mis_form_id = flow_ids.get(flow_form_id)
    
    if parent_mis_form_id is None:
        print(f"No MIS form mapping found for Flow form ID {flow_form_id}")
        continue
    
    # Find parent form using predefined ID
    parent_form = find_mis_form_by_id(parent_mis_form_id)
    
    if parent_form is None:
        print(f"No parent MIS form found with ID {parent_mis_form_id}")
        continue
    
    print(f"Parent MIS form: {parent_form['name']} (ID: {parent_form['id']})")
    
    # Get child form IDs from flow_mis_child_forms mapping
    child_form_ids = flow_mis_child_forms.get(flow_form_id, [])
    
    # Find child forms using predefined IDs
    child_forms = []
    for child_id in child_form_ids:
        child_form = find_mis_form_by_id(child_id)
        if child_form:
            child_forms.append(child_form)
    
    print(f"Found {len(child_forms)} child forms (from {len(child_form_ids)} predefined IDs)")
    for child in child_forms:
        print(f"  - {child['name']} (ID: {child['id']})")
    
    # Load parent form questions
    parent_mis_questions_df = flatten_flow_form_pd(
        json_path=parent_form['file'],
        record_path="question_groups.questions",
        meta_type="mis",
    )
    
    # Normalize question text for matching (do once for Flow questions)
    flow_questions_df["_norm"] = flow_questions_df.get("text", "").apply(_normalize_alpha_only)
    flow_questions_df["_group_norm"] = flow_questions_df.get("flow_question_group", "").apply(_normalize_alpha_only)
    
    # --- Match Flow questions against PARENT form only ---
    parent_mis_questions_df["_norm"] = parent_mis_questions_df.get("label", "").apply(_normalize_alpha_only)
    parent_mis_questions_df["_group_norm"] = parent_mis_questions_df.get("mis_question_group", "").apply(_normalize_alpha_only)
    
    print(f"\n--- Matching Flow questions against PARENT form ---")
    parent_mapping_df = match_questions_by_text(flow_questions_df, parent_mis_questions_df)
    
    # Load existing manual matches for parent (if any)
    parent_output_csv = output_dir / f"{flow_form_id}_mapping_parent_{parent_form['id']}.csv"
    parent_manual_df = load_existing_manual_matches(parent_output_csv)
    
    if not parent_manual_df.empty:
        print(f"Found {len(parent_manual_df)} existing manual matches in parent CSV")
        parent_mapping_df = merge_manual_matches(parent_mapping_df, parent_manual_df)
    
    if not parent_mapping_df.empty:
        # Convert float IDs to integers before saving
        parent_mapping_df = convert_float_ids_to_int(parent_mapping_df, ID_COLUMNS)
        parent_mapping_df.to_csv(parent_output_csv, index=False)
        manual_count = len(parent_mapping_df[parent_mapping_df['match_method'] == 'manual'])
        auto_count = len(parent_mapping_df[parent_mapping_df['match_method'] == 'text_similarity'])
        print(f"Saved parent mapping: {parent_output_csv} ({len(parent_mapping_df)} rows)")
        print(f"  - Auto-matched: {auto_count}, Manual: {manual_count}")
    else:
        print(f"No parent mappings found for {flow_form_id}")
    
    # --- Match Flow questions against EACH child form SEPARATELY ---
    # This allows the same Flow question to match multiple child forms
    # (e.g., both Monitoring and Quick Monitoring which have similar questions)
    child_mappings_summary = []
    for child in child_forms:
        print(f"\n--- Matching Flow questions against child form: {child['name']} ---")
        
        # Load this child's questions
        child_mis_questions_df = flatten_flow_form_pd(
            json_path=child['file'],
            record_path="question_groups.questions",
            meta_type="mis",
        )
        
        # Normalize for this child
        child_mis_questions_df["_norm"] = child_mis_questions_df.get("label", "").apply(_normalize_alpha_only)
        child_mis_questions_df["_group_norm"] = child_mis_questions_df.get("mis_question_group", "").apply(_normalize_alpha_only)
        
        # Match Flow questions to THIS child form only
        child_mapping_df = match_questions_by_text(flow_questions_df, child_mis_questions_df)
        
        # Load existing manual matches for this child (if any)
        child_output_csv = output_dir / f"{flow_form_id}_mapping_child_{child['id']}.csv"
        child_manual_df = load_existing_manual_matches(child_output_csv)
        
        if not child_manual_df.empty:
            print(f"Found {len(child_manual_df)} existing manual matches in child CSV")
            child_mapping_df = merge_manual_matches(child_mapping_df, child_manual_df)
        
        if not child_mapping_df.empty:
            # Convert float IDs to integers before saving
            child_mapping_df = convert_float_ids_to_int(child_mapping_df, ID_COLUMNS)
            child_mapping_df.to_csv(child_output_csv, index=False)
            manual_count = len(child_mapping_df[child_mapping_df['match_method'] == 'manual'])
            auto_count = len(child_mapping_df[child_mapping_df['match_method'] == 'text_similarity'])
            child_mappings_summary.append((child['name'], len(child_mapping_df), child_output_csv.name, auto_count, manual_count))
            print(f"Saved child mapping ({child['name']}): {child_output_csv} ({len(child_mapping_df)} rows)")
            print(f"  - Auto-matched: {auto_count}, Manual: {manual_count}")
        else:
            child_mappings_summary.append((child['name'], 0, f"{flow_form_id}_mapping_child_{child['id']}.csv", 0, 0))
            print(f"No mappings found for child form: {child['name']} (ID: {child['id']})")
    
    # Print summary
    print(f"\n=== Summary for Flow Form {flow_form_id} ===")
    print(f"Total mappings: {len(parent_mapping_df)}")
    print(f"  - Parent mappings: {len(parent_mapping_df)}")
    for name, count, filename, auto_count, manual_count in child_mappings_summary:
        print(f"  - Child mappings ({name}): {count} (auto: {auto_count}, manual: {manual_count})")
    print(f"\nOutput files:")
    print(f"  - Parent: {flow_form_id}_mapping_parent_{parent_form['id']}.csv")
    for name, count, filename, auto_count, manual_count in child_mappings_summary:
        if count > 0:
            print(f"  - Child: {filename}")